## Bibliotecas

In [40]:
import pandas as pd

## Definindo bases

In [41]:
ses_gruposramos = pd.read_csv(
    '/home/gbocalon/Projects/youse-estrategia-de-clientes/analise_mkt_sizing/bases_susep/ses_gruposramos.csv',
    sep=';',
    encoding='latin-1'  # ou 'cp1252'
)

ses_ramos = pd.read_csv(
    '/home/gbocalon/Projects/youse-estrategia-de-clientes/analise_mkt_sizing/bases_susep/Ses_ramos.csv',
    sep=';',
    encoding='latin-1'  # ou 'cp1252'
)

ses_uf = pd.read_csv(
    '/home/gbocalon/Projects/youse-estrategia-de-clientes/analise_mkt_sizing/bases_susep/SES_UF2.csv',
    sep=';',
    encoding='latin-1',
    decimal=',',
    low_memory=False  # também resolve o DtypeWarning de colunas com tipos mistos
)

KeyboardInterrupt: 

## Exploração

In [ ]:
ses_ramos.head()

,coramo,noramo
0,1101,1101 - Seguro Agr sem cob do FESR(RUN OFF)
1,1601,1601 - Microsseguros de Pessoas
2,2201,2201 - Sobrevivência de Assistido
3,1102,1102 - Seguro Agr com cob do FESR(RUN OFF)
4,1602,1602 - Microsseguros de Danos


In [ ]:
print(ses_ramos['noramo'].unique())
# print('2293 - Vida EFPC' in ses_ramos['noramo'].unique())

<StringArray>
['1101 - Seguro Agr sem cob do FESR(RUN OFF)',
            '1601 - Microsseguros de Pessoas',
          '2201 - Sobrevivência de Assistido',
 '1102 - Seguro Agr com cob do FESR(RUN OFF)',
              '1602 - Microsseguros de Danos',
                    '2202 - Fluxo Biométrico',
 '1103 - Seguro Pec sem cob do FESR(RUN OFF)',
         '1603 - Microsseguros - Previdência',
                   '2203 - Índice Biométrico',
 '1104 - Seguro Pec com cob do FESR(RUN OFF)',
 ...
   '0994 - VGBL/ VAGP/ VRGP/ VRSA/ VRID/ VDR',
 '0195 - Garantia Est./Ext.Gar-Bens em Geral',
      '0196 - Riscos Nomeados e Operacionais',
            '0996 - Seguro de Vida Universal',
            '1396 - Seguro de Vida Universal',
                              '0997 - VG/APC',
 '1597 - Resp. Explor. ou Transp. Aéreo-RETA',
    '1198 - Seguro de Vida do Produtor Rural',
      '1299 - SUCURSAIS NO EXTERIOR(RUN OFF)',
                '2199 - SUCURSAL NO EXTERIOR']
Length: 161, dtype: str


In [ ]:
print(ses_gruposramos)

    GRAID                      GRANOME  GRACODIGO
0       1             01 - Patrimonial          1
1       2        02 - Riscos Especiais          2
2       3       03 - Responsabilidades          3
3       4                  04 - Cascos          4
4       5               05 - Automóvel          5
5       6             06 - Transportes          6
6       7      07 - Riscos Financeiros          7
7       8                 08 - Crédito          8
8       9        09 - Pessoas Coletivo          9
9      10            10 - Habitacional         10
10     11                   11 - RURAL         11
11     12                  12 - Outros         12
12     13       13- Pessoas Individual         13
13     14               14 - Marítimos         14
14     16           16 - Microsseguros         16
15     21    21 - Sucursal no Exterior         21
16     20  20 - Aceitações do Exterior         20
17     19                   19 - Saúde         19
18     18               18 - Nucleares         18


In [ ]:
ses_uf.head()

,coenti,damesano,ramos,UF,premio_dir,premio_ret,sin_dir,prem_ret_liq,gracodigo,salvados,recuperacao
0,6785,200811,435,RN,0.00,0.00,0.00,0.00,4,0.0,0.0
1,6921,200507,553,MG,0.00,0.00,239.86,0.00,5,0.0,0.0
2,8737,200511,1066,SC,0.00,0.00,0.00,0.00,10,0.0,0.0
3,6785,200501,977,TO,55553.86,55553.86,-60000.00,55553.86,9,0.0,0.0
4,6785,200612,980,MS,0.00,0.00,0.00,0.00,9,0.0,0.0


In [ ]:
# Gerar todos as strings de damesano de 202501 até 202512 (incluindo todos os meses)
periodos = [f"2025{str(m).zfill(2)}" for m in range(1, 13)]
resultado = ses_uf[
    (ses_uf['coenti'] == 5886) &
    (ses_uf['damesano'].astype(str).isin(periodos))
]['premio_dir'].sum()
print(resultado)

22254778134.75


## Pré-processamento — base filtrada para ramos Youse

Carrega `Ses_cias`, filtra `SES_UF2` para os ramos do escopo Youse e cria as colunas derivadas usadas em todas as análises.

> **Período rolling 12M:** mai/2025 – abr/2026 (último mês disponível = abr/2026)

In [54]:
ses_cias = pd.read_csv(
    '/home/gbocalon/Projects/youse-estrategia-de-clientes/analise_mkt_sizing/bases_susep/Ses_cias.csv',
    sep=';', encoding='latin-1'
)
ses_cias.columns = ses_cias.columns.str.strip()
ses_cias['Coenti'] = ses_cias['Coenti'].astype(str).str.strip().astype(int)
ses_cias['Noenti'] = ses_cias['Noenti'].str.strip()

# Definições do escopo Youse (replicadas aqui para garantir execução independente)
RAMOS_YOUSE = {
    "Auto":        [531, 553, 542, 520, 525, 527],
    "Residencial": [114, 116, 112, 113],
    "Vida":        [1391, 1396, 1381, 1384, 1387, 1390],
}
TODOS_RAMOS_YOUSE = [r for ramos in RAMOS_YOUSE.values() for r in ramos]
PRODUTO_POR_RAMO  = {ramo: prod for prod, ramos in RAMOS_YOUSE.items() for ramo in ramos}

MACRO_REGIAO = {
    'AC':'Norte','AM':'Norte','AP':'Norte','PA':'Norte','RO':'Norte','RR':'Norte','TO':'Norte',
    'AL':'Nordeste','BA':'Nordeste','CE':'Nordeste','MA':'Nordeste','PB':'Nordeste',
    'PE':'Nordeste','PI':'Nordeste','RN':'Nordeste','SE':'Nordeste',
    'DF':'Centro-Oeste','GO':'Centro-Oeste','MS':'Centro-Oeste','MT':'Centro-Oeste',
    'ES':'Sudeste','MG':'Sudeste','RJ':'Sudeste','SP':'Sudeste',
    'PR':'Sul','RS':'Sul','SC':'Sul',
}

PERIODO_ROLLING = (
    [f"2025{m:02d}" for m in range(5, 13)] +
    [f"2026{m:02d}" for m in range(1, 5)]
)

# Filtra SES_UF2 para os ramos do escopo Youse
df = ses_uf[ses_uf['ramos'].isin(TODOS_RAMOS_YOUSE)].copy()

# Colunas de valor: converte para float caso estejam como string com vírgula decimal
for col in ['premio_dir', 'sin_dir', 'premio_ret', 'prem_ret_liq']:
    if df[col].dtype == object:
        df[col] = df[col].str.replace(',', '.', regex=False).astype(float)

df['produto']     = df['ramos'].map(PRODUTO_POR_RAMO)
df['macro_regiao'] = df['UF'].str.strip().map(MACRO_REGIAO).fillna('Outros')
df['ano']         = df['damesano'].astype(str).str[:4].astype(int)
df['mes_ano']     = df['damesano'].astype(str).str.zfill(6).str[:6]

df_anual   = df[df['ano'].between(2016, 2026)]
df_rolling = df[df['mes_ano'].isin(PERIODO_ROLLING)]

print(f"Linhas no escopo (2016–2026): {len(df_anual):,}")
print(f"Linhas no rolling 12M:        {len(df_rolling):,}")
print(f"Empresas únicas (rolling):    {df_rolling['coenti'].nunique()}")

Linhas no escopo (2016–2026): 878,151
Linhas no rolling 12M:        103,363
Empresas únicas (rolling):    106


## Análise 1 — Tamanho Total do Mercado (2016–2026)

Evolução anual de prêmios diretos e sinistros diretos do mercado nos ramos Youse (Auto + Residencial + Vida), com CAGR e loss ratio.

In [44]:
a1 = (
    df_anual
    .groupby('ano', as_index=False)
    .agg(premio_dir=('premio_dir', 'sum'), sin_dir=('sin_dir', 'sum'))
)
a1['loss_ratio'] = a1['sin_dir'] / a1['premio_dir']
a1['premio_B']   = a1['premio_dir'] / 1e9
a1['sinistro_B'] = a1['sin_dir']   / 1e9
a1['lr_pct']     = a1['loss_ratio'] * 100

# CAGR 2016–2025 (anos completos)
p2016 = a1.loc[a1['ano'] == 2016, 'premio_dir'].values[0]
p2025 = a1.loc[a1['ano'] == 2025, 'premio_dir'].values[0]
cagr  = (p2025 / p2016) ** (1/9) - 1

print("Mercado total (ramos Youse) — Prêmio Direto e Sinistralidade")
print(f"{'Ano':<6}{'Prêmio (R$B)':>14}{'Sinistro (R$B)':>16}{'Loss Ratio':>12}")
print("-" * 50)
for _, r in a1.iterrows():
    ytd = " (YTD)" if r['ano'] == 2026 else ""
    print(f"{int(r['ano']):<6}{r['premio_B']:>12.1f}B{r['sinistro_B']:>14.1f}B{r['lr_pct']:>11.1f}%{ytd}")

print(f"\nCAGR Prêmio 2016–2025: {cagr*100:.1f}% a.a.")
print(f"Loss Ratio 2025: {a1.loc[a1['ano']==2025,'lr_pct'].values[0]:.1f}%")

# DataFrame para ThinkCell (exportável)
a1_export = a1[a1['ano'] <= 2025][['ano','premio_B','sinistro_B','lr_pct']].copy()
a1_export.columns = ['Ano','Premio_B','Sinistro_B','LossRatio_pct']
a1_export

Mercado total (ramos Youse) — Prêmio Direto e Sinistralidade
Ano     Prêmio (R$B)  Sinistro (R$B)  Loss Ratio
--------------------------------------------------
2016          38.1B          23.2B       60.8%
2017          41.0B          23.5B       57.4%
2018          44.3B          23.7B       53.5%
2019          47.3B          24.7B       52.2%
2020          48.5B          22.6B       46.5%
2021          54.6B          29.1B       53.3%
2022          70.6B          36.7B       52.0%
2023          79.0B          35.3B       44.7%
2024          85.6B          40.3B       47.1%
2025          93.5B          43.4B       46.4%
2026          31.2B          15.4B       49.4% (YTD)

CAGR Prêmio 2016–2025: 10.5% a.a.
Loss Ratio 2025: 46.4%


,Ano,Premio_B,Sinistro_B,LossRatio_pct
0,2016,38.071893,23.161227,60.835502
1,2017,41.007704,23.521569,57.358901
2,2018,44.299442,23.690721,53.478598
3,2019,47.276473,24.698762,52.243242
4,2020,48.517768,22.573404,46.526055
5,2021,54.624841,29.109465,53.289794
6,2022,70.559534,36.685664,51.992498
7,2023,79.003203,35.342039,44.734945
8,2024,85.623992,40.308373,47.076027
9,2025,93.517659,43.398862,46.407131


### Key Findings — Análise 1

**1. Mercado em forte expansão: CAGR de 10,5% a.a. (2016–2025)**
O mercado combinado de Auto + Residencial + Vida saltou de R$ 38,1B em 2016 para R$ 93,5B em 2025 — mais que dobrou em 9 anos. O crescimento acelerou a partir de 2021, com saltos anuais acima de R$ 10B.

**2. Sinistralidade em queda estrutural: de 60,8% para 46,4%**
O loss ratio caiu 14,4 p.p. em 9 anos, sinalizando que o setor ficou mais eficiente tecnicamente. A recuperação de 2020 foi mais acentuada (pandemia reduziu sinistros de auto), mas mesmo após o rebote de 2021–2022 o patamar não voltou ao nível histórico de 2016.

**3. Ponto de atenção: aceleração de sinistros em 2021–2022**
O sinistro saltou de R$ 22,6B (2020) para R$ 36,7B (2022), crescendo mais rápido que os prêmios (+62% vs. +45%). O mercado respondeu com reprecificação que sustentou a queda do LR nos anos seguintes.

**4. 2026 no ritmo certo**
Os 4 primeiros meses de 2026 somam R$ 31,3B em prêmios — ritmo que, anualizado, projeta ~R$ 94B, em linha com a tendência de crescimento de 10%+ a.a.

> **Gráfico ThinkCell sugerido:** Combo chart — barras agrupadas (prêmio + sinistro em R$B) + linha secundária (loss ratio em %). Bracket de CAGR de 2016 a 2025 sobre a barra de prêmio.

## Análise 2 — Drill Down de Prêmios (Rolling 12M: mai/2025–abr/2026)

Composição do mercado de prêmios diretos no último ano móvel, por produto e por macro-região.

In [45]:
total_rolling = df_rolling['premio_dir'].sum()

# Por produto
a2_produto = (
    df_rolling
    .groupby('produto', as_index=False)['premio_dir'].sum()
    .sort_values('premio_dir', ascending=False)
)
a2_produto['share_pct'] = a2_produto['premio_dir'] / total_rolling * 100
a2_produto['premio_B']  = a2_produto['premio_dir'] / 1e9

# Por macro-região
a2_regiao = (
    df_rolling
    .groupby('macro_regiao', as_index=False)['premio_dir'].sum()
    .sort_values('premio_dir', ascending=False)
)
a2_regiao['share_pct'] = a2_regiao['premio_dir'] / total_rolling * 100
a2_regiao['premio_B']  = a2_regiao['premio_dir'] / 1e9

print(f"Prêmio Direto Total (mai/25–abr/26): R$ {total_rolling/1e9:.2f}B\n")

print("Por Produto:")
print(f"{'Produto':<15}{'Prêmio (R$B)':>14}{'Share':>8}")
print("-" * 38)
for _, r in a2_produto.iterrows():
    print(f"{r['produto']:<15}{r['premio_B']:>12.2f}B{r['share_pct']:>7.1f}%")

print("\nPor Macro-Região:")
print(f"{'Região':<15}{'Prêmio (R$B)':>14}{'Share':>8}")
print("-" * 38)
for _, r in a2_regiao.iterrows():
    print(f"{r['macro_regiao']:<15}{r['premio_B']:>12.2f}B{r['share_pct']:>7.1f}%")

# DataFrames para ThinkCell
print("\n--- Exportável ThinkCell (por produto) ---")
display(a2_produto[['produto','premio_B','share_pct']].rename(columns={'produto':'Produto','premio_B':'Premio_B','share_pct':'Share_pct'}))

print("\n--- Exportável ThinkCell (por região) ---")
display(a2_regiao[['macro_regiao','premio_B','share_pct']].rename(columns={'macro_regiao':'Regiao','premio_B':'Premio_B','share_pct':'Share_pct'}))

Prêmio Direto Total (mai/25–abr/26): R$ 95.87B

Por Produto:
Produto          Prêmio (R$B)   Share
--------------------------------------
auto                  62.58B   65.3%
vida                  25.19B   26.3%
residencial            8.09B    8.4%

Por Macro-Região:
Região           Prêmio (R$B)   Share
--------------------------------------
Sudeste               57.88B   60.4%
Sul                   18.63B   19.4%
Nordeste               9.85B   10.3%
Centro-Oeste           7.19B    7.5%
Norte                  2.32B    2.4%

--- Exportável ThinkCell (por produto) ---


,Produto,Premio_B,Share_pct
0,auto,62.584573,65.280006
2,vida,25.194583,26.279679
1,residencial,8.091812,8.440315



--- Exportável ThinkCell (por região) ---


,Regiao,Premio_B,Share_pct
3,Sudeste,57.878886,60.371651
4,Sul,18.633904,19.436440
1,Nordeste,9.852624,10.276963
0,Centro-Oeste,7.186287,7.495791
2,Norte,2.319268,2.419155


### Key Findings — Análise 2

**1. Auto é 65% do mercado — e é onde a Youse tem seu produto âncora**
Auto soma R$ 62,6B dos R$ 95,9B totais. Residencial (8,4% / R$ 8,1B) e Vida (26,3% / R$ 25,2B) são mercados menores, mas representam oportunidades de cross-sell com menor sinistralidade (ver Análise 3).

**2. Sudeste concentra 60% do mercado — mas Sul cresce acima da média**
R$ 57,9B estão no Sudeste (SP é o driver principal). Sul com 19,4% (R$ 18,6B) é o 2º maior mercado. Norte (2,4%) representa o menor mercado com potencial de expansão.

**3. Residencial é o menor mercado absoluto, mas com LR mais saudável**
Com apenas R$ 8,1B em prêmios, o segmento residencial tem espaço para crescimento — especialmente considerando o baixo loss ratio (22,9%, ver Análise 3) que o torna atrativo para expansão de carteira.

**4. Concentração geográfica: Sudeste + Sul = ~80% do mercado**
O mercado é altamente concentrado nas regiões mais ricas — padrão esperado, mas que evidencia oportunidade de penetração nas regiões Centro-Oeste (7,5%) e Nordeste (10,3%).

> **Gráfico ThinkCell sugerido:** Pie/donut para produto (3 fatias). Horizontal bar rankeado por macro-região com % acumulado. Ambos na mesma página.

## Análise 3 — Sinistralidade por Produto e Região (Rolling 12M: mai/2025–abr/2026)

Loss ratio = sinistro direto / prêmio direto, por produto e por macro-região.

In [53]:
tot_p_roll = df_rolling['premio_dir'].sum()
tot_s_roll = df_rolling['sin_dir'].sum()
lr_total   = tot_s_roll / tot_p_roll * 100

# Por produto
a3_produto = (
    df_rolling
    .groupby('produto', as_index=False)
    .agg(premio_dir=('premio_dir','sum'), sin_dir=('sin_dir','sum'))
)
a3_produto['loss_ratio_pct'] = a3_produto['sin_dir'] / a3_produto['premio_dir'] * 100
a3_produto['premio_B']       = a3_produto['premio_dir'] / 1e9
a3_produto['sinistro_B']     = a3_produto['sin_dir']   / 1e9
a3_produto = a3_produto.sort_values('loss_ratio_pct', ascending=False)

# Por macro-região
a3_regiao = (
    df_rolling
    .groupby('macro_regiao', as_index=False)
    .agg(premio_dir=('premio_dir','sum'), sin_dir=('sin_dir','sum'))
)
a3_regiao['loss_ratio_pct'] = a3_regiao['sin_dir'] / a3_regiao['premio_dir'] * 100
a3_regiao['premio_B']       = a3_regiao['premio_dir'] / 1e9
a3_regiao['sinistro_B']     = a3_regiao['sin_dir']   / 1e9
a3_regiao = a3_regiao.sort_values('loss_ratio_pct', ascending=False)

print(f"Loss Ratio Total do Mercado (mai/25–abr/26): {lr_total:.1f}%\n")

print("Por Produto:")
print(f"{'Produto':<15}{'LR':>8}{'Prêmio(B)':>12}{'Sinistro(B)':>13}")
print("-" * 50)
for _, r in a3_produto.iterrows():
    print(f"{r['produto']:<15}{r['loss_ratio_pct']:>7.1f}%{r['premio_B']:>10.2f}B{r['sinistro_B']:>11.2f}B")

print("\nPor Macro-Região:")
print(f"{'Região':<15}{'LR':>8}{'Prêmio(B)':>12}{'Sinistro(B)':>13}")
print("-" * 50)
for _, r in a3_regiao.iterrows():
    print(f"{r['macro_regiao']:<15}{r['loss_ratio_pct']:>7.1f}%{r['premio_B']:>10.2f}B{r['sinistro_B']:>11.2f}B")

# DataFrames para ThinkCell
print("\n--- Exportável ThinkCell (por produto) ---")
display(a3_produto[['produto','loss_ratio_pct','premio_B','sinistro_B']].rename(
    columns={'produto':'Produto','loss_ratio_pct':'LossRatio_pct','premio_B':'Premio_B','sinistro_B':'Sinistro_B'}))

print("\n--- Exportável ThinkCell (por região) ---")
display(a3_regiao[['macro_regiao','loss_ratio_pct','premio_B','sinistro_B']].rename(
    columns={'macro_regiao':'Regiao','loss_ratio_pct':'LossRatio_pct','premio_B':'Premio_B','sinistro_B':'Sinistro_B'}))

Loss Ratio Total do Mercado (mai/25–abr/26): 46.9%

Por Produto:
Produto              LR   Prêmio(B)  Sinistro(B)
--------------------------------------------------
auto              65.2%     62.58B      40.82B
residencial       22.9%      8.09B       1.86B
vida               8.9%     25.19B       2.25B

Por Macro-Região:
Região               LR   Prêmio(B)  Sinistro(B)
--------------------------------------------------
Sul               53.7%     18.63B      10.01B
Centro-Oeste      52.0%      7.19B       3.74B
Nordeste          46.7%      9.85B       4.60B
Norte             44.9%      2.32B       1.04B
Sudeste           44.1%     57.88B      25.53B

--- Exportável ThinkCell (por produto) ---


,Produto,LossRatio_pct,Premio_B,Sinistro_B
0,auto,65.221869,62.584573,40.818829
1,residencial,22.928370,8.091812,1.855321
2,vida,8.916861,25.194583,2.246566



--- Exportável ThinkCell (por região) ---


,Regiao,LossRatio_pct,Premio_B,Sinistro_B
4,Sul,53.700272,18.633904,10.006457
0,Centro-Oeste,52.025387,7.186287,3.738694
1,Nordeste,46.711579,9.852624,4.602316
2,Norte,44.892186,2.319268,1.041170
3,Sudeste,44.112940,57.878886,25.532078


### Key Findings — Análise 3

**1. Auto tem LR de 65,2% — mais que o triplo de Residencial e Vida combinados**
A sinistralidade de Auto (65,2%) é structuralmente muito maior que Residencial (22,9%) e Vida (8,9%). Isso significa que a Youse, com Auto como produto âncora, opera no segmento de maior risco do seu portfólio. Cada ponto percentual de melhora no LR de Auto tem impacto desproporcional no resultado técnico.

**2. Vida com LR de apenas 8,9% — produto com melhor perfil de resultado**
O seguro de vida individual é o produto de menor sinistralidade do portfólio. Isso reflete a natureza de proteção de longo prazo, com sinistros infrequentes e bem precificados. Oportunidade de expansão com baixo risco técnico.

**3. Sul e Centro-Oeste são as regiões de maior sinistralidade**
Sul (53,7%) e Centro-Oeste (52,0%) têm LR acima da média nacional (46,9%), enquanto Sudeste (44,1%) — que concentra 60% dos prêmios — fica abaixo da média. Isso cria uma tensão geográfica: crescer fora do Sudeste significa assumir mais risco.

**4. Spread de LR entre produtos revela oportunidade de mix**
Auto representa 65% do prêmio mas tem o pior LR. Se a Youse conseguir aumentar a participação de Residencial e Vida na base (cross-sell), o LR médio da carteira cai estruturalmente — mesmo sem melhora na precificação de Auto.

> **Gráfico ThinkCell sugerido:** Bar chart horizontal com os 3 produtos e as 5 regiões, cada um com sua barra de LR. Linha de referência vertical no LR total do mercado (46,9%). Cores diferenciando "acima da média" vs. "abaixo da média".

## Análise 4 — Concentração de Mercado / Top 10 Competidores (Rolling 12M: mai/2025–abr/2026)

Ranking das maiores seguradoras por prêmio direto em cada segmento, com market share e loss ratio individual.

In [55]:
a4_base = (
    df_rolling
    .groupby(['coenti', 'produto'], as_index=False)
    .agg(premio_dir=('premio_dir','sum'), sin_dir=('sin_dir','sum'))
)
a4_base = a4_base.merge(ses_cias[['Coenti','Noenti']], left_on='coenti', right_on='Coenti', how='left')
a4_base['noenti']         = a4_base['Noenti'].fillna('Empresa ' + a4_base['coenti'].astype(str))
a4_base['loss_ratio_pct'] = a4_base['sin_dir'] / a4_base['premio_dir'] * 100
a4_base['premio_B']       = a4_base['premio_dir'] / 1e9

for produto in ['Auto', 'Residencial', 'Vida']:
    df_prod  = a4_base[a4_base['produto'] == produto].sort_values('premio_dir', ascending=False)
    tot      = df_prod['premio_dir'].sum()
    top10    = df_prod.head(10).copy()
    top10['share_pct'] = top10['premio_dir'] / tot * 100
    top10['share_acum'] = top10['share_pct'].cumsum()
    top10_share = top10['premio_dir'].sum() / tot * 100
    n_empresas  = len(df_prod)

    print(f"\n{'='*75}")
    print(f"  {produto.upper()} — Mercado total: R$ {tot/1e9:.2f}B | {n_empresas} empresas ativas")
    print(f"{'='*75}")
    print(f"{'#':<3}{'Empresa':<44}{'Prêmio':>9}{'Share':>7}{'Acum.':>7}{'LR':>7}")
    print("-" * 75)
    for i, (_, r) in enumerate(top10.iterrows(), 1):
        nome = r['noenti'][:43]
        print(f"{i:<3}{nome:<44}{r['premio_B']:>7.2f}B{r['share_pct']:>6.1f}%{r['share_acum']:>6.1f}%{r['loss_ratio_pct']:>6.1f}%")
    print(f"\n  >> Top 10 concentra {top10_share:.1f}% do mercado | 'Outros' = {100-top10_share:.1f}%")

    display(top10[['noenti','premio_B','share_pct','share_acum','loss_ratio_pct']].rename(columns={
        'noenti':'Empresa','premio_B':'Premio_B','share_pct':'Share_pct',
        'share_acum':'Share_Acumulado','loss_ratio_pct':'LossRatio_pct'
    }).reset_index(drop=True))


  AUTO — Mercado total: R$ 62.58B | 57 empresas ativas
#  Empresa                                        Prêmio  Share  Acum.     LR
---------------------------------------------------------------------------
1  PORTO SEGURO COMPANHIA DE SEGUROS GERAIS      16.24B  26.0%  26.0%  53.5%
2  TOKIO MARINE SEGURADORA S.A.                   9.04B  14.4%  40.4%  63.1%
3  ALLIANZ SEGUROS S.A.                           8.89B  14.2%  54.6%  69.3%
4  BRADESCO AUTO/RE COMPANHIA DE SEGUROS          6.99B  11.2%  65.8%  71.6%
5  YELUM SEGUROS S.A.                             5.76B   9.2%  75.0%  71.8%
6  HDI SEGUROS S.A.                               4.49B   7.2%  82.1%  73.7%
7  MAPFRE SEGUROS GERAIS S.A.                     3.26B   5.2%  87.3%  53.2%
8  SUHAI SEGURADORA S.A.                          1.98B   3.2%  90.5%  64.1%
9  ZURICH MINAS BRASIL SEGUROS S.A.               1.60B   2.6%  93.1%  71.5%
10 SANTANDER AUTO S.A.                            0.71B   1.1%  94.2%  17.5%

  >> Top 10 concent

,Empresa,Premio_B,Share_pct,Share_Acumulado,LossRatio_pct
0,PORTO SEGURO COMPANHIA DE SEGUROS GERAIS,16.241765,25.951706,25.951706,53.528699
1,TOKIO MARINE SEGURADORA S.A.,9.038451,14.441979,40.393685,63.121032
2,ALLIANZ SEGUROS S.A.,8.894837,14.212507,54.606192,69.349786
3,BRADESCO AUTO/RE COMPANHIA DE SEGUROS,6.985663,11.161957,65.768150,71.557740
4,YELUM SEGUROS S.A.,5.756308,9.197647,74.965796,71.761815
5,HDI SEGUROS S.A.,4.489672,7.173768,82.139564,73.703343
6,MAPFRE SEGUROS GERAIS S.A.,3.256045,5.202633,87.342197,53.202198
7,SUHAI SEGURADORA S.A.,1.976675,3.158406,90.500603,64.096014
8,ZURICH MINAS BRASIL SEGUROS S.A.,1.602963,2.561275,93.061878,71.502930
9,SANTANDER AUTO S.A.,0.706529,1.128918,94.190796,17.480861



  RESIDENCIAL — Mercado total: R$ 8.09B | 63 empresas ativas
#  Empresa                                        Prêmio  Share  Acum.     LR
---------------------------------------------------------------------------
1  PORTO SEGURO COMPANHIA DE SEGUROS GERAIS       1.55B  19.1%  19.1%  30.0%
2  BRADESCO AUTO/RE COMPANHIA DE SEGUROS          1.15B  14.2%  33.3%  16.9%
3  XS3 SEGUROS S.A.                               1.10B  13.6%  46.9%   3.4%
4  ALLIANZ SEGUROS S.A.                           0.78B   9.7%  56.6%  37.6%
5  TOKIO MARINE SEGURADORA S.A.                   0.72B   8.9%  65.6%  29.4%
6  HDI SEGUROS S.A.                               0.54B   6.6%  72.2%  49.4%
7  ALIANÇA DO BRASIL SEGUROS S.A                  0.51B   6.3%  78.5%  17.0%
8  ZURICH SANTANDER BRASIL SEGUROS S.A.           0.48B   5.9%  84.5%   7.5%
9  CHUBB SEGUROS BRASIL S.A.                      0.23B   2.8%  87.3%  12.6%
10 MAPFRE SEGUROS GERAIS S.A.                     0.21B   2.6%  89.9%  30.3%

  >> Top 10 c

,Empresa,Premio_B,Share_pct,Share_Acumulado,LossRatio_pct
0,PORTO SEGURO COMPANHIA DE SEGUROS GERAIS,1.546224,19.108507,19.108507,29.987278
1,BRADESCO AUTO/RE COMPANHIA DE SEGUROS,1.149433,14.204894,33.313401,16.853756
2,XS3 SEGUROS S.A.,1.103134,13.632721,46.946122,3.382837
3,ALLIANZ SEGUROS S.A.,0.783071,9.677327,56.623449,37.578725
4,TOKIO MARINE SEGURADORA S.A.,0.723844,8.945385,65.568834,29.406718
5,HDI SEGUROS S.A.,0.537036,6.636789,72.205623,49.383178
6,ALIANÇA DO BRASIL SEGUROS S.A,0.513163,6.341762,78.547386,16.967894
7,ZURICH SANTANDER BRASIL SEGUROS S.A.,0.479041,5.920074,84.467460,7.537152
8,CHUBB SEGUROS BRASIL S.A.,0.229799,2.839901,87.307361,12.550787
9,MAPFRE SEGUROS GERAIS S.A.,0.210883,2.606132,89.913493,30.311884



  VIDA — Mercado total: R$ 25.19B | 76 empresas ativas
#  Empresa                                        Prêmio  Share  Acum.     LR
---------------------------------------------------------------------------
1  BRADESCO VIDA E PREVIDÊNCIA S.A.               9.46B  37.5%  37.5%   3.1%
2  PRUDENTIAL DO BRASIL SEGUROS S.A.              6.85B  27.2%  64.7%  10.1%
3  MONGERAL AEGON SEGUROS E PREVIDÊNCIA S. A.     1.57B   6.2%  71.0%   5.3%
4  METROPOLITAN LIFE SEGUROS E PREVIDÊNCIA        1.54B   6.1%  77.1%   4.4%
5  BRASILSEG COMPANHIA DE SEGUROS                 1.14B   4.5%  81.6%  10.9%
6  PORTO SEGURO COMPANHIA DE SEGUROS GERAIS       0.78B   3.1%  84.7%  37.6%
7  ICATU SEGUROS S.A                              0.63B   2.5%  87.2%   5.9%
8  SICOOB SEGURADORA DE VIDA E PREVIDÊNCIA S.A    0.51B   2.0%  89.2%  28.7%
9  ITAU SEGUROS S.A.                              0.33B   1.3%  90.5%  14.3%
10 CAIXA VIDA E PREVIDÊNCIA S.A.                  0.31B   1.2%  91.8%   7.3%

  >> Top 10 concent

,Empresa,Premio_B,Share_pct,Share_Acumulado,LossRatio_pct
0,BRADESCO VIDA E PREVIDÊNCIA S.A.,9.458735,37.542731,37.542731,3.128101
1,PRUDENTIAL DO BRASIL SEGUROS S.A.,6.845606,27.170946,64.713677,10.093449
2,MONGERAL AEGON SEGUROS E PREVIDÊNCIA S. A.,1.571385,6.236996,70.950673,5.339991
3,METROPOLITAN LIFE SEGUROS E PREVIDÊNCIA,1.540556,6.114630,77.065303,4.386757
4,BRASILSEG COMPANHIA DE SEGUROS,1.144887,4.544178,81.609481,10.948565
5,PORTO SEGURO COMPANHIA DE SEGUROS GERAIS,0.779097,3.092318,84.701799,37.576446
6,ICATU SEGUROS S.A,0.631375,2.505996,87.207795,5.899036
7,SICOOB SEGURADORA DE VIDA E PREVIDÊNCIA S.A.,0.506666,2.011012,89.218807,28.718288
8,ITAU SEGUROS S.A.,0.331435,1.315501,90.534308,14.328442
9,CAIXA VIDA E PREVIDÊNCIA S.A.,0.314826,1.249576,91.783884,7.292406


### Key Findings — Análise 4

#### Auto (R$ 62,6B | 57 empresas | Top 10 = 94,2%)

**1. Porto Seguro domina com 26% — mas tem um dos menores LR do setor (53,5%)**
Porto Seguro lidera com R$ 16,2B e apresenta sinistralidade significativamente mais baixa que os concorrentes diretos (Allianz: 69,3%; Bradesco: 71,6%; HDI: 73,7%). Isso indica vantagem de seleção de risco ou precificação mais eficiente — referência de benchmark para a Youse.

**2. Top 3 concentra 54,6% — mercado altamente oligopolizado**
Porto (26%) + Tokio Marine (14,4%) + Allianz (14,2%) somam mais da metade do mercado. A Youse compete num mercado onde escala e reputação de marca têm grande peso.

**3. Spread de LR entre líderes: 17,5% (Santander) a 73,7% (HDI)**
O Santander Auto tem LR anormalmente baixo (17,5%), provavelmente por ser um produto de nicho bancário com perfil de risco selecionado. O spread amplo indica que precificação e seleção de risco são diferenciais críticos.

---

#### Residencial (R$ 8,1B | 63 empresas | Top 10 = 89,9%)

**4. Mercado mais fragmentado que Auto — top player tem "apenas" 19,1%**
Porto Seguro lidera com 19,1%, mas o mercado é mais disputado. XS3 Seguros (13,6%) com LR de 3,4% chama atenção — provavelmente produto bancário de baixo risco que funciona como fee business.

**5. LR muito baixo em grande parte dos players (< 20%)**
Bradesco (16,9%), XS3 (3,4%), Aliança do Brasil (17%), Zurich Santander (7,5%): players bancários têm LR extremamente baixo. Seguradoras de varejo como HDI (49,4%) e Porto (30%) têm perfil mais parecido com o da Youse.

---

#### Vida (R$ 25,2B | 76 empresas | Top 10 = 91,8%)

**6. Bradesco + Prudential = 64,7% do mercado de Vida Individual**
O mercado de vida individual é ainda mais concentrado no topo que auto. Bradesco (37,5%) e Prudential (27,2%) dominam com LR muito baixo (3,1% e 10,1%). São players com décadas de carteira estabelecida.

**7. LR médio do setor Vida é de apenas ~8%**
Os maiores players têm LR entre 3% e 11%, com exceção de Porto (37,6%) e Sicoob (28,7%). Isso revela que o seguro de vida individual é altamente rentável tecnicamente — e explica por que é o produto com maior atratividade para expansão.

> **Gráfico ThinkCell sugerido:** Para cada produto, um horizontal bar chart rankeado (Top 10 por prêmio) com a barra colorida pelo LR (gradiente: verde = baixo LR, vermelho = alto LR). Ou um waterfall mostrando acumulação de market share do #1 ao #10 + "Outros".

In [ ]:
# Gerar todos as strings de damesano de 202501 até 202512 (incluindo todos os meses)
periodos = [f"2025{str(m).zfill(2)}" for m in range(1, 13)]

resultado = ses_uf[
    (ses_uf['damesano'].astype(str).isin(periodos))
]['premio_dir'].sum()


print(resultado)

362601621490.54553


## Escopo da análise — Produtos Youse

Define os grupos de ramos e ramos relevantes para a operação da Youse, com base nos 4 produtos comercializados:
**Seguro Auto**, **Seguro Residencial**, **Seguro Vida** e **Seguro Auto por Km**.

> A relação grupo de ramos ↔ ramo é determinada pelos 2 primeiros dígitos do código `coramo` = `gracodigo` (validado em 100% das ~10M linhas de `SES_UF2`).  
> Ramos com `(RUN OFF)` no nome foram excluídos — representam carteiras encerradas sem novos negócios.  
> Documentação completa em `documentation/escopo_ramos_youse.md`.

In [49]:
# Grupos de ramos por produto Youse
# gracodigo armazenado como inteiro em SES_UF2
GRACODIGOS_YOUSE = {
    "Auto":        5,   # 05 - Automóvel
    "Residencial": 1,   # 01 - Patrimonial
    "Vida":        13,  # 13 - Pessoas Individual
}

# Ramos ativos (sem RUN OFF) por produto
# coramo armazenado como inteiro em SES_UF2 (ex.: "0531" → 531)
RAMOS_YOUSE = {
    "Auto": [
        531,   # Automóvel - Casco              → cobertura principal de danos ao veículo
        553,   # R. C. Facultativa Veículos      → cobertura de RC ao terceiro
        542,   # Assistência e Outras Coberturas → assistência 24h e coberturas acessórias
        520,   # Acidentes Pessoais Passageiros  → cobertura de ocupantes do veículo
        525,   # Carta Verde                     → RC obrigatória MERCOSUL
        527,   # RC Veic Pass Acordos fora Mercosul
    ],
    "Residencial": [
        114,   # Compreensivo Residencial        → produto principal Youse Reside
        116,   # Compreensivo Condomínio         → complementar
        112,   # Assistência - Bens em Geral     → complementar
        113,   # Vidros                          → complementar
    ],
    "Vida": [
        1391,  # Vida                            → produto principal
        1396,  # Seguro de Vida Universal        → produto principal
        1381,  # Acidentes Pessoais              → complementar
        1384,  # Doenças Graves ou Doença Terminal → complementar
        1387,  # Desemprego/Perda de Renda       → complementar
        1390,  # Eventos Aleatórios              → complementar
    ],
}

# Listas planas para filtros diretos em DataFrames
TODOS_GRACODIGOS_YOUSE = list(GRACODIGOS_YOUSE.values())              # [5, 1, 13]
TODOS_RAMOS_YOUSE      = [r for ramos in RAMOS_YOUSE.values() for r in ramos]

# De/para: ramo → produto (para enriquecer DataFrames com label de produto)
PRODUTO_POR_RAMO = {
    ramo: produto
    for produto, ramos in RAMOS_YOUSE.items()
    for ramo in ramos
}

print("Grupos de ramos Youse:", TODOS_GRACODIGOS_YOUSE)
print("Ramos Youse:", TODOS_RAMOS_YOUSE)
print(f"Total de ramos no escopo: {len(TODOS_RAMOS_YOUSE)}")

Grupos de ramos Youse: [5, 1, 13]
Ramos Youse: [531, 553, 542, 520, 525, 527, 114, 116, 112, 113, 1391, 1396, 1381, 1384, 1387, 1390]
Total de ramos no escopo: 16
